# Task 05：Policy Gradient——学习与实验入口

## 本 Notebook 的学习边界

策略梯度直接优化 $\pi_	heta(a\mid s)$。REINFORCE 的单轨迹目标可写为：

$$

abla_	heta J(	heta)pprox\sum_t G_t
abla_	heta\log\pi_	heta(A_t\mid S_t).
$$

先理解 policy network 和采样，再运行 vanilla REINFORCE，随后加入 value baseline，最后用多个 seed 比较均值、方差和达到阈值所需 episode 数。


In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.optim as optim

import sys

sys.path.append("../src")

from policy_network import PolicyNetwork
from reinforce import update_policy
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print("state_dim:", state_dim)
print("action_dim:", action_dim)

state_dim: 4
action_dim: 2


In [5]:
policy = PolicyNetwork(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=128,
)

optimizer = optim.Adam(
    policy.parameters(),
    lr=1e-3,
)

state, info = env.reset(seed=SEED)

log_probs = []
rewards = []

terminated = False
truncated = False

while not (terminated or truncated):

    state_tensor = torch.tensor(
        state,
        dtype=torch.float32,
    )

    action, log_prob = policy.sample_action(
        state_tensor
    )

    next_state, reward, terminated, truncated, info = env.step(
        action.item()
    )

    log_probs.append(log_prob)
    rewards.append(reward)

    state = next_state
    


loss, returns = update_policy(
    optimizer=optimizer,
    log_probs=log_probs,
    rewards=rewards,
    gamma=0.99,
    normalize_returns=True,
)

print("Episode length:", len(rewards))
print("Episode return:", sum(rewards))
print("Policy loss:", loss)

Episode length: 17
Episode return: 17.0
Policy loss: 0.6867489814758301


In [6]:
def train_reinforce(
    env,
    policy,
    optimizer,
    num_episodes=1000,
    gamma=0.99,
):
    episode_returns = []
    episode_losses = []

    for episode in range(num_episodes):

        state, info = env.reset()

        log_probs = []
        rewards = []

        terminated = False
        truncated = False

        while not (terminated or truncated):

            state_tensor = torch.tensor(
                state,
                dtype=torch.float32,
            )

            action, log_prob = policy.sample_action(
                state_tensor
            )

            next_state, reward, terminated, truncated, info = env.step(
                action.item()
            )

            log_probs.append(log_prob)
            rewards.append(reward)

            state = next_state

        loss, _ = update_policy(
            optimizer=optimizer,
            log_probs=log_probs,
            rewards=rewards,
            gamma=gamma,
            normalize_returns=True,
        )

        episode_return = sum(rewards)

        episode_returns.append(episode_return)
        episode_losses.append(loss)

        if (episode + 1) % 50 == 0:

            avg_return = np.mean(
                episode_returns[-50:]
            )

            print(
                f"Episode {episode + 1:4d} | "
                f"Return {episode_return:6.1f} | "
                f"Avg Return {avg_return:6.1f}"
            )

    return episode_returns, episode_losses

episode_returns, episode_losses = train_reinforce(
    env=env,
    policy=policy,
    optimizer=optimizer,
    num_episodes=1000,
    gamma=0.99,
)

Episode   50 | Return   56.0 | Avg Return   27.5
Episode  100 | Return   18.0 | Avg Return   31.2
Episode  150 | Return   33.0 | Avg Return   40.0
Episode  200 | Return   39.0 | Avg Return   48.5
Episode  250 | Return  170.0 | Avg Return   76.5
Episode  300 | Return  317.0 | Avg Return  135.4
Episode  350 | Return  141.0 | Avg Return  182.2
Episode  400 | Return  322.0 | Avg Return  254.1
Episode  450 | Return  225.0 | Avg Return  353.7
Episode  500 | Return  500.0 | Avg Return  372.3
Episode  550 | Return  331.0 | Avg Return  459.3
Episode  600 | Return   85.0 | Avg Return  187.9
Episode  650 | Return  500.0 | Avg Return  360.9
Episode  700 | Return  500.0 | Avg Return  470.1
Episode  750 | Return  500.0 | Avg Return  458.9
Episode  800 | Return  500.0 | Avg Return  490.8
Episode  850 | Return  500.0 | Avg Return  475.1
Episode  900 | Return  500.0 | Avg Return  489.0
Episode  950 | Return  500.0 | Avg Return  473.4
Episode 1000 | Return  361.0 | Avg Return  489.1


## 实验结论与提交要求

运行完本 Notebook 后，不要只保留图。请在对应 `notes/` 中记录随机种子、环境版本、关键超参数、最终指标、曲线文件和一个失败现象。结论必须区分“代码运行成功”和“算法表现更好”：前者由自检确认，后者需要多 seed 或控制变量实验支持。

提交前从仓库根目录运行：

```bash
python eval/run.py
```
